In [27]:
import os

current_directory = os.getcwd()
print(f"Current working directory: {current_directory}")


Current working directory: c:\Users\shreyaan.saha\Desktop\Projects\Actual Projs\BrightSpeed


In [28]:
# =========================================================
# FINAL CHURN PIPELINE WITH PREPROCESSING - STRICT VERSION
# GRIDSEARCHCV + EXPANDING WALK-FORWARD CV VERSION
# =========================================================
#
# BUSINESS / MODELING DESIGN
# ---------------------------------------------------------
# 1. Filter to Copper DSL only
# 2. Define churn target as:
#       churn in next 1 to 3 months = 1
#       otherwise = 0
# 3. Use strict 12-month feature history discipline:
#       a snapshot month is eligible only if full 12 months
#       of historical data exist before / through that month
# 4. Use only ACTIVE customers at snapshot for modeling
# 5. Apply minimum tenure rule
# 6. Keep rolling 3m / 6m / 12m features
# 7. Use expanding walk-forward validation on snapshot months
# 8. Keep final untouched OOS months
# 9. Run GridSearchCV using ONLY pre-OOS months and ONLY custom
#    time-based expanding folds
# 10. Train final best model on all pre-OOS snapshots
# 11. Score latest active eligible snapshot in production style
# 12. Add business ranking metrics:
#       - decile table
#       - recall @ top 10%
#       - recall @ top 20%
#       - lift @ top 10%
#       - lift @ top 20%
#
# IMPORTANT MODELING LOGIC
# ---------------------------------------------------------
# - One row = one account at one snapshot month
# - Features are built using information up to that snapshot month
# - Label looks into the NEXT 3 months
# - Only customers active at snapshot are eligible
# - Snapshots used for train/val/test must have full future 3-month
#   label availability
#
# PREPROCESSING
# ---------------------------------------------------------
# Learned on TRAIN only and applied to validation / OOS / latest:
# - Winsorization
# - Variance threshold
# - Optional VIF filtering (only if enabled, normally logistic only)
# - Scaling (only for Logistic Regression)
#
# GRID SEARCH DESIGN
# ---------------------------------------------------------
# This implementation uses sklearn GridSearchCV BUT DOES NOT use random CV.
# Instead, GridSearchCV receives custom expanding time-based folds.
#
# This preserves:
# - telecom-safe temporal validation
# - train-only preprocessing
# - no leakage
#
# NOTE
# ---------------------------------------------------------
# This code includes important hyperparameters for each model family,
# including imbalance handling:
# - Logistic Regression -> C, class_weight
# - Random Forest      -> n_estimators, max_depth, min_samples_leaf, class_weight
# - LightGBM           -> n_estimators, learning_rate, num_leaves,
#                         min_child_samples, subsample, colsample_bytree,
#                         class_weight
# - XGBoost            -> n_estimators, learning_rate, max_depth,
#                         min_child_weight, subsample, colsample_bytree,
#                         scale_pos_weight
# =========================================================

import json
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score
)
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import GridSearchCV, ParameterGrid

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


# =========================================================
# STEP 0: CONFIGURATION
# =========================================================
INPUT_FILE = "Brightspeed_Synthetic_Churn_KPI_Monthly_AllColumns (1).xlsx"
INPUT_SHEET = "monthly_model_table"

COPPER_VALUE = "Copper_DSL"

FEATURE_HISTORY_MONTHS = 12
LABEL_HORIZON_MONTHS = 3

MIN_TENURE_MONTHS = 6

MIN_TRAIN_SNAPSHOT_COUNT = 5
N_OOS_MONTHS = 2

TARGET_COL = "churn_flag_3m"

# Classification threshold used for threshold-based metrics
CLASSIFICATION_THRESHOLD = 0.03


# =========================================================
# STEP 1: LOAD DATA
# =========================================================
df = pd.read_excel(
    INPUT_FILE,
    sheet_name=INPUT_SHEET
)

df["snapshot_month"] = pd.to_datetime(df["snapshot_month"], errors="coerce")
df["activation_date"] = pd.to_datetime(df["activation_date"], errors="coerce")
df["commitment_end_date"] = pd.to_datetime(df["commitment_end_date"], errors="coerce")

df = df.dropna(subset=["snapshot_month"]).copy()

print("Raw dataset shape:", df.shape)


# =========================================================
# STEP 2: FILTER TO COPPER DSL ONLY
# =========================================================
df = df[df["service_type"] == COPPER_VALUE].copy()
df = df.sort_values(["account_number", "snapshot_month"]).reset_index(drop=True)

print("Filtered dataset shape (Copper only):", df.shape)
print("Unique service types after filter:", df["service_type"].unique())


# =========================================================
# STEP 3: CREATE CHURN EVENT MONTH
# =========================================================
df["lifecycle_stage_lower"] = df["lifecycle_stage"].astype(str).str.lower()

churn_month_map = (
    df.loc[df["lifecycle_stage_lower"] == "disconnected"]
      .groupby("account_number")["snapshot_month"]
      .min()
)

df["churn_event_month"] = df["account_number"].map(churn_month_map)


# =========================================================
# STEP 4: CREATE MONTHS_TO_CHURN
# =========================================================
df["months_to_churn"] = np.where(
    df["churn_event_month"].notna(),
    (
        (df["churn_event_month"].dt.year - df["snapshot_month"].dt.year) * 12
        + (df["churn_event_month"].dt.month - df["snapshot_month"].dt.month)
    ),
    np.nan
)


# =========================================================
# STEP 5: CREATE 3-MONTH FUTURE CHURN LABEL
# =========================================================
df[TARGET_COL] = np.where(
    (df["months_to_churn"] >= 1) & (df["months_to_churn"] <= LABEL_HORIZON_MONTHS),
    1,
    0
)

print("\nTarget distribution on full filtered table:")
print(df[TARGET_COL].value_counts(dropna=False))


# =========================================================
# STEP 6: FEATURE ENGINEERING
# =========================================================

# ---------------------------------------------------------
# 6.1 Structural / lifecycle features
# ---------------------------------------------------------
df["tenure_months"] = (
    (df["snapshot_month"] - df["activation_date"]).dt.days / 30
).clip(lower=0)

df["months_to_commitment_end"] = (
    (df["commitment_end_date"] - df["snapshot_month"]).dt.days / 30
)
df["months_to_commitment_end"] = df["months_to_commitment_end"].fillna(-999)

df["near_contract_end_flag"] = np.where(
    (df["months_to_commitment_end"] >= 0) &
    (df["months_to_commitment_end"] <= 3),
    1,
    0
)

df["competitive_pressure"] = (
    df["fiber_available_at_premises"] +
    df["competitor_broadband_available_by_address"]
)

# ---------------------------------------------------------
# 6.2 History coverage features
# ---------------------------------------------------------
df["history_available_months"] = (
    df.groupby("account_number").cumcount() + 1
).clip(upper=FEATURE_HISTORY_MONTHS)

df["history_coverage_ratio_12m"] = (
    df["history_available_months"] / FEATURE_HISTORY_MONTHS
).clip(upper=1.0)

df["new_customer_flag"] = np.where(df["tenure_months"] < 12, 1, 0)

# ---------------------------------------------------------
# 6.3 Current / snapshot-level derived features
# ---------------------------------------------------------
df["prev_bill"] = df.groupby("account_number")["bill_amount"].shift(1)

df["bill_shock_flag"] = np.where(
    df["bill_amount"] > 1.2 * df["prev_bill"],
    1,
    0
)
df["bill_shock_flag"] = df["bill_shock_flag"].fillna(0)

df["throughput_to_speed_ratio"] = (
    df["avg_delivered_throughput_mbps"] / df["subscribed_speed_mbps"]
)
df["throughput_to_speed_ratio"] = (
    df["throughput_to_speed_ratio"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

df["speed_gap_ratio"] = (
    (df["subscribed_speed_mbps"] - df["avg_delivered_throughput_mbps"]) /
    df["subscribed_speed_mbps"]
)
df["speed_gap_ratio"] = (
    df["speed_gap_ratio"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
    .clip(lower=0)
)

df["revenue_per_speed"] = (
    df["mrc_data"] / df["subscribed_speed_mbps"]
)
df["revenue_per_speed"] = (
    df["revenue_per_speed"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

df["outage_severity"] = (
    df["network_outage_events"] * (df["network_outage_duration_minutes"] / 60)
)
df["outage_severity"] = df["outage_severity"].fillna(0)

# ---------------------------------------------------------
# 6.4 Rolling 3-month features
# ---------------------------------------------------------
df["tickets_last_3m_sum"] = (
    df.groupby("account_number")["trouble_ticket_volume"]
      .rolling(3, min_periods=1)
      .sum()
      .reset_index(level=0, drop=True)
)

df["outages_last_3m_sum"] = (
    df.groupby("account_number")["network_outage_events"]
      .rolling(3, min_periods=1)
      .sum()
      .reset_index(level=0, drop=True)
)

df["bill_volatility_3m_std"] = (
    df.groupby("account_number")["bill_amount"]
      .rolling(3, min_periods=1)
      .std()
      .reset_index(level=0, drop=True)
).fillna(0)

df["nps_3m_min"] = (
    df.groupby("account_number")["nps_score"]
      .rolling(3, min_periods=1)
      .min()
      .reset_index(level=0, drop=True)
)

df["csat_3m_min"] = (
    df.groupby("account_number")["csat_score"]
      .rolling(3, min_periods=1)
      .min()
      .reset_index(level=0, drop=True)
)

df["usage_3m_avg"] = (
    df.groupby("account_number")["data_consumption_gb"]
      .rolling(3, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

df["tickets_3m_avg"] = (
    df.groupby("account_number")["trouble_ticket_volume"]
      .rolling(3, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

# ---------------------------------------------------------
# 6.5 Rolling 6-month features
# ---------------------------------------------------------
df["tickets_last_6m_sum"] = (
    df.groupby("account_number")["trouble_ticket_volume"]
      .rolling(6, min_periods=1)
      .sum()
      .reset_index(level=0, drop=True)
)

df["outages_last_6m_sum"] = (
    df.groupby("account_number")["network_outage_events"]
      .rolling(6, min_periods=1)
      .sum()
      .reset_index(level=0, drop=True)
)

df["bill_volatility_6m_std"] = (
    df.groupby("account_number")["bill_amount"]
      .rolling(6, min_periods=1)
      .std()
      .reset_index(level=0, drop=True)
).fillna(0)

df["nps_6m_min"] = (
    df.groupby("account_number")["nps_score"]
      .rolling(6, min_periods=1)
      .min()
      .reset_index(level=0, drop=True)
)

df["csat_6m_min"] = (
    df.groupby("account_number")["csat_score"]
      .rolling(6, min_periods=1)
      .min()
      .reset_index(level=0, drop=True)
)

df["usage_6m_avg"] = (
    df.groupby("account_number")["data_consumption_gb"]
      .rolling(6, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

# ---------------------------------------------------------
# 6.6 Rolling 12-month features
# ---------------------------------------------------------
df["tickets_last_12m_sum"] = (
    df.groupby("account_number")["trouble_ticket_volume"]
      .rolling(12, min_periods=1)
      .sum()
      .reset_index(level=0, drop=True)
)

df["outages_last_12m_sum"] = (
    df.groupby("account_number")["network_outage_events"]
      .rolling(12, min_periods=1)
      .sum()
      .reset_index(level=0, drop=True)
)

df["bill_volatility_12m_std"] = (
    df.groupby("account_number")["bill_amount"]
      .rolling(12, min_periods=1)
      .std()
      .reset_index(level=0, drop=True)
).fillna(0)

df["nps_12m_min"] = (
    df.groupby("account_number")["nps_score"]
      .rolling(12, min_periods=1)
      .min()
      .reset_index(level=0, drop=True)
)

df["csat_12m_min"] = (
    df.groupby("account_number")["csat_score"]
      .rolling(12, min_periods=1)
      .min()
      .reset_index(level=0, drop=True)
)

df["usage_12m_avg"] = (
    df.groupby("account_number")["data_consumption_gb"]
      .rolling(12, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

df["bill_shock_last_12m_sum"] = (
    df.groupby("account_number")["bill_shock_flag"]
      .rolling(12, min_periods=1)
      .sum()
      .reset_index(level=0, drop=True)
)

df["outage_severity_last_12m_sum"] = (
    df.groupby("account_number")["outage_severity"]
      .rolling(12, min_periods=1)
      .sum()
      .reset_index(level=0, drop=True)
)

# ---------------------------------------------------------
# 6.7 Trend / acceleration features
# ---------------------------------------------------------
df["usage_mom_pct_change"] = (
    df.groupby("account_number")["data_consumption_gb"]
      .pct_change()
      .replace([np.inf, -np.inf], 0)
      .fillna(0)
)

df["usage_drop_ratio_3m_vs_12m"] = (
    (df["usage_3m_avg"] - df["usage_12m_avg"]) /
    df["usage_12m_avg"].replace(0, np.nan)
).replace([np.inf, -np.inf], 0).fillna(0)

df["ticket_acceleration_3m_vs_12m"] = (
    df["tickets_3m_avg"] - (df["tickets_last_12m_sum"] / 12.0)
)

df["outage_acceleration_3m_vs_12m"] = (
    (df["outages_last_3m_sum"] / 3.0) - (df["outages_last_12m_sum"] / 12.0)
)

df["bill_volatility_acceleration_3m_vs_12m"] = (
    df["bill_volatility_3m_std"] - df["bill_volatility_12m_std"]
)

df["ticket_per_month"] = (
    df["trouble_ticket_volume"] / df["tenure_months"].replace(0, np.nan)
)
df["ticket_per_month"] = df["ticket_per_month"].replace([np.inf, -np.inf], 0).fillna(0)

# ---------------------------------------------------------
# 6.8 Exposure-normalized recent history feature
# ---------------------------------------------------------
df["ticket_per_active_month_12m"] = (
    df["tickets_last_12m_sum"] / df["history_available_months"].replace(0, np.nan)
).replace([np.inf, -np.inf], 0).fillna(0)

# ---------------------------------------------------------
# 6.9 Tenure bucket
# ---------------------------------------------------------
df["tenure_bucket"] = pd.cut(
    df["tenure_months"],
    bins=[0, 6, 18, 36, 48, np.inf],
    labels=["new_customer", "early_growth", "established", "long_term", "very_long_term"],
    include_lowest=True
)

le = LabelEncoder()
df["tenure_bucket_encoded"] = le.fit_transform(df["tenure_bucket"].astype(str))

# ---------------------------------------------------------
# 6.10 Additional frustration / spike features
# ---------------------------------------------------------
df["ticket_spike_ratio"] = (
    df["tickets_last_3m_sum"] /
    df["tickets_last_6m_sum"].replace(0, np.nan)
).replace([np.inf, -np.inf], np.nan).fillna(0)

df["outage_trend_ratio"] = (
    df["outages_last_3m_sum"] /
    df["outages_last_6m_sum"].replace(0, np.nan)
).replace([np.inf, -np.inf], np.nan).fillna(0)

df["usage_last_3m"] = (
    df.groupby("account_number")["data_consumption_gb"]
      .rolling(3, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

df["usage_last_6m"] = (
    df.groupby("account_number")["data_consumption_gb"]
      .rolling(6, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

df["usage_drop_ratio"] = (
    df["usage_last_3m"] /
    df["usage_last_6m"].replace(0, np.nan)
).replace([np.inf, -np.inf], np.nan).fillna(1)

df["fiber_speed_risk"] = (
    df["fiber_available_at_premises"] *
    df["speed_gap_ratio"]
)

df["service_frustration_index"] = (
    df["ticket_spike_ratio"] +
    df["outage_trend_ratio"] +
    df["speed_gap_ratio"]
)

df["frustration_acceleration"] = (
    df["service_frustration_index"] -
    df.groupby("account_number")["service_frustration_index"].shift(3)
).fillna(0)


# =========================================================
# STEP 7: FINAL FEATURE LIST
# =========================================================
feature_cols = [
    "fiber_available_at_premises",
    "competitor_broadband_available_by_address",
    "competitive_pressure",

    "tenure_months",
    "tenure_bucket_encoded",
    "new_customer_flag",
    "history_available_months",
    "history_coverage_ratio_12m",

    "months_to_commitment_end",
    "near_contract_end_flag",

    "network_outage_events",
    "trouble_ticket_volume",
    "repeat_issue_flag",
    "promo_expiration_flag",
    "price_increase_flag",
    "late_payment_flag",
    "collections_activity_flag",
    "bill_shock_flag",
    # "throughput_to_speed_ratio",
    "speed_gap_ratio",
    "revenue_per_speed",
    "outage_severity",

    "tickets_last_3m_sum",
    "outages_last_3m_sum",
    "bill_volatility_3m_std",
    "nps_3m_min",
    "csat_3m_min",

    "tickets_last_6m_sum",
    "outages_last_6m_sum",
    "bill_volatility_6m_std",
    "nps_6m_min",
    "csat_6m_min",

    "tickets_last_12m_sum",
    "outages_last_12m_sum",
    "bill_volatility_12m_std",
    "nps_12m_min",
    "csat_12m_min",
    "bill_shock_last_12m_sum",
    "outage_severity_last_12m_sum",

    "usage_mom_pct_change",
    "usage_drop_ratio_3m_vs_12m",
    "ticket_acceleration_3m_vs_12m",
    "outage_acceleration_3m_vs_12m",
    "bill_volatility_acceleration_3m_vs_12m",

    "ticket_per_active_month_12m",

    "ticket_spike_ratio",
    "outage_trend_ratio",
    "usage_drop_ratio",
    "fiber_speed_risk",
    "service_frustration_index",
    "usage_3m_avg",
    "usage_6m_avg",
    "usage_12m_avg",
    "tickets_3m_avg",
    "frustration_acceleration",
    "ticket_per_month"
]


# =========================================================
# STEP 8: BUILD STRICT MODELING BASE
# =========================================================
all_months_full = sorted(pd.to_datetime(df["snapshot_month"].dropna().unique()))
all_months_full = [pd.Timestamp(m) for m in all_months_full]

if len(all_months_full) < (FEATURE_HISTORY_MONTHS + LABEL_HORIZON_MONTHS):
    raise ValueError(
        "Not enough monthly history to support strict 12-month feature window "
        "and 3-month label window."
    )

first_full_history_snapshot = all_months_full[FEATURE_HISTORY_MONTHS - 1]
last_labelable_snapshot = all_months_full[-(LABEL_HORIZON_MONTHS + 1)]

print("\nFirst full-history snapshot month:", first_full_history_snapshot.date())
print("Last labelable snapshot month:", last_labelable_snapshot.date())

model_df = df[
    (df["lifecycle_stage_lower"] == "active") &
    (df["tenure_months"] >= MIN_TENURE_MONTHS) &
    (df["snapshot_month"] >= first_full_history_snapshot) &
    (df["snapshot_month"] <= last_labelable_snapshot)
].copy()

model_df = model_df.sort_values(["snapshot_month", "account_number"]).reset_index(drop=True)

eligible_snapshot_months = sorted(model_df["snapshot_month"].unique())
eligible_snapshot_months = [pd.Timestamp(m) for m in eligible_snapshot_months]

print("\nEligible modeling snapshot months:")
for m in eligible_snapshot_months:
    print(m.date())

print("\nTotal eligible snapshot months:", len(eligible_snapshot_months))
print("Modeling base shape:", model_df.shape)

print("\nTarget distribution in strict modeling base:")
print(model_df[TARGET_COL].value_counts(dropna=False))


# =========================================================
# STEP 9: BUILD EXPANDING WALK-FORWARD FOLDS
# =========================================================
def build_expanding_folds(month_list, min_train_months=6, n_oos_months=2):
    month_list = sorted(month_list)

    if len(month_list) < (min_train_months + 1 + n_oos_months):
        raise ValueError(
            "Not enough eligible snapshot months for requested CV + OOS design."
        )

    pretest_months = month_list[:-n_oos_months]
    oos_months = month_list[-n_oos_months:]

    folds = []
    for i in range(min_train_months, len(pretest_months)):
        train_months = pretest_months[:i]
        val_month = pretest_months[i]

        folds.append({
            "train_months": train_months,
            "val_months": [val_month]
        })

    return folds, pretest_months, oos_months


cv_folds, pretest_months, oos_months = build_expanding_folds(
    eligible_snapshot_months,
    min_train_months=MIN_TRAIN_SNAPSHOT_COUNT,
    n_oos_months=N_OOS_MONTHS
)

print("\nExpanding walk-forward CV folds:")
for i, fold in enumerate(cv_folds, 1):
    print(
        f"Fold {i} | "
        f"Train: {fold['train_months'][0].date()} -> {fold['train_months'][-1].date()} | "
        f"Validate: {fold['val_months'][0].date()}"
    )

print("\nPre-test months:", pretest_months[0].date(), "->", pretest_months[-1].date())
print("Final OOS months:", oos_months[0].date(), "->", oos_months[-1].date())


# =========================================================
# STEP 10: HELPER FUNCTIONS
# =========================================================
def get_snapshot_slice(data, months):
    return data[data["snapshot_month"].isin(months)].copy()


def make_model_matrix(data, feature_cols, target_col=None):
    X = data[feature_cols].copy()

    if target_col is not None:
        y = data[target_col].copy()
        return X, y

    return X


# =========================================================
# STEP 11: PREPROCESSING HELPERS
# =========================================================
def fit_winsor_bounds(X_train, lower_q=0.05, upper_q=0.95):
    bounds = {}
    for col in X_train.columns:
        lower = X_train[col].quantile(lower_q)
        upper = X_train[col].quantile(upper_q)
        bounds[col] = (lower, upper)
    return bounds


def apply_winsor_bounds(X, bounds):
    X = X.copy()
    for col, (lower, upper) in bounds.items():
        if col in X.columns:
            X[col] = X[col].clip(lower=lower, upper=upper)
    return X


def remove_high_vif_features(X, threshold=10):
    try:
        from statsmodels.stats.outliers_influence import variance_inflation_factor
    except ImportError:
        print("statsmodels not installed; skipping VIF filtering.")
        return X

    X = X.copy()

    nunique = X.nunique()
    constant_cols = nunique[nunique <= 1].index.tolist()
    if constant_cols:
        X = X.drop(columns=constant_cols)

    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

    while X.shape[1] > 1:
        vif_df = pd.DataFrame()
        vif_df["feature"] = X.columns
        vif_df["VIF"] = [
            variance_inflation_factor(X.values, i)
            for i in range(X.shape[1])
        ]

        max_vif = vif_df["VIF"].max()

        if pd.isna(max_vif):
            break

        if max_vif > threshold:
            drop_feature = vif_df.sort_values("VIF", ascending=False)["feature"].iloc[0]
            print(f"Dropping {drop_feature} due to high VIF: {max_vif:.2f}")
            X = X.drop(columns=[drop_feature])
        else:
            break

    return X


def fit_preprocessing_pipeline(
    X_train,
    use_scaling=False,
    use_vif=False,
    vif_threshold=10,
    winsorize=True,
    lower_q=0.05,
    upper_q=0.95,
    variance_threshold=0.00
):
    artifacts = {}

    X_train_proc = X_train.copy()
    X_train_proc = X_train_proc.replace([np.inf, -np.inf], np.nan).fillna(0)

    artifacts["raw_train_columns"] = X_train_proc.columns.tolist()

    if winsorize:
        winsor_bounds = fit_winsor_bounds(X_train_proc, lower_q=lower_q, upper_q=upper_q)
        X_train_proc = apply_winsor_bounds(X_train_proc, winsor_bounds)
        artifacts["winsor_bounds"] = winsor_bounds
    else:
        artifacts["winsor_bounds"] = None

    vt = VarianceThreshold(threshold=variance_threshold)
    X_train_proc = pd.DataFrame(
        vt.fit_transform(X_train_proc),
        columns=X_train_proc.columns[vt.get_support()],
        index=X_train_proc.index
    )
    artifacts["variance_selector"] = vt
    artifacts["post_variance_columns"] = X_train_proc.columns.tolist()

    if use_vif:
        X_train_proc = remove_high_vif_features(X_train_proc, threshold=vif_threshold)

    artifacts["final_columns"] = X_train_proc.columns.tolist()

    scaler = None
    if use_scaling:
        scaler = StandardScaler()
        X_train_model = scaler.fit_transform(X_train_proc)
    else:
        X_train_model = X_train_proc

    artifacts["scaler"] = scaler

    return X_train_proc, X_train_model, artifacts


def apply_preprocessing_pipeline(X, artifacts, use_scaling=False):
    X_proc = X.copy()
    X_proc = X_proc.replace([np.inf, -np.inf], np.nan).fillna(0)

    if artifacts["winsor_bounds"] is not None:
        X_proc = apply_winsor_bounds(X_proc, artifacts["winsor_bounds"])

    X_proc = X_proc.reindex(columns=artifacts["raw_train_columns"], fill_value=0)

    vt = artifacts["variance_selector"]
    X_proc = pd.DataFrame(
        vt.transform(X_proc),
        columns=artifacts["post_variance_columns"],
        index=X_proc.index
    )

    X_proc = X_proc.reindex(columns=artifacts["final_columns"], fill_value=0)

    if use_scaling and artifacts["scaler"] is not None:
        X_model = artifacts["scaler"].transform(X_proc)
    else:
        X_model = X_proc

    return X_proc, X_model


# =========================================================
# STEP 12: CUSTOM ESTIMATOR FOR GRIDSEARCHCV
# =========================================================
#
# PURPOSE
# ---------------------------------------------------------
# GridSearchCV needs an sklearn-style estimator.
# This wrapper ensures:
# - preprocessing is learned ONLY on train fold
# - same preprocessing is applied to validation / OOS / latest
# - model family can switch between logistic / rf / lightgbm / xgboost
#
# IMPORTANT
# ---------------------------------------------------------
# This avoids using standard GridSearchCV with raw shuffled CV.
# We still use your custom expanding time-based folds.
# =========================================================
class ChurnPreprocessedEstimator(BaseEstimator, ClassifierMixin):
    def __init__(
        self,
        model_family="logistic",

        # Common preprocessing knobs
        winsorize=True,
        lower_q=0.05,
        upper_q=0.95,
        variance_threshold=0.00,
        use_vif=False,
        vif_threshold=10,

        # Logistic params
        C=1.0,
        logistic_class_weight="balanced",
        penalty="l2",

        # Random Forest params
        rf_n_estimators=200,
        rf_max_depth=6,
        rf_min_samples_leaf=10,
        rf_class_weight="balanced",
        rf_max_features="sqrt",

        # LightGBM params
        lgbm_n_estimators=200,
        lgbm_learning_rate=0.05,
        lgbm_num_leaves=31,
        lgbm_min_child_samples=20,
        lgbm_subsample=0.8,
        lgbm_colsample_bytree=0.8,
        lgbm_class_weight="balanced",

        # XGBoost params
        xgb_n_estimators=200,
        xgb_learning_rate=0.05,
        xgb_max_depth=4,
        xgb_min_child_weight=1,
        xgb_subsample=0.8,
        xgb_colsample_bytree=0.8,
        xgb_scale_pos_weight=20
    ):
        self.model_family = model_family

        self.winsorize = winsorize
        self.lower_q = lower_q
        self.upper_q = upper_q
        self.variance_threshold = variance_threshold
        self.use_vif = use_vif
        self.vif_threshold = vif_threshold

        self.C = C
        self.logistic_class_weight = logistic_class_weight
        self.penalty = penalty

        self.rf_n_estimators = rf_n_estimators
        self.rf_max_depth = rf_max_depth
        self.rf_min_samples_leaf = rf_min_samples_leaf
        self.rf_class_weight = rf_class_weight
        self.rf_max_features = rf_max_features

        self.lgbm_n_estimators = lgbm_n_estimators
        self.lgbm_learning_rate = lgbm_learning_rate
        self.lgbm_num_leaves = lgbm_num_leaves
        self.lgbm_min_child_samples = lgbm_min_child_samples
        self.lgbm_subsample = lgbm_subsample
        self.lgbm_colsample_bytree = lgbm_colsample_bytree
        self.lgbm_class_weight = lgbm_class_weight

        self.xgb_n_estimators = xgb_n_estimators
        self.xgb_learning_rate = xgb_learning_rate
        self.xgb_max_depth = xgb_max_depth
        self.xgb_min_child_weight = xgb_min_child_weight
        self.xgb_subsample = xgb_subsample
        self.xgb_colsample_bytree = xgb_colsample_bytree
        self.xgb_scale_pos_weight = xgb_scale_pos_weight

    def _build_model(self):
        if self.model_family == "logistic":
            model = LogisticRegression(
                max_iter=1000,
                C=self.C,
                class_weight=self.logistic_class_weight,
                penalty=self.penalty,
                random_state=42
            )
            scale_flag = True

        elif self.model_family == "rf":
            model = RandomForestClassifier(
                n_estimators=self.rf_n_estimators,
                max_depth=self.rf_max_depth,
                min_samples_leaf=self.rf_min_samples_leaf,
                class_weight=self.rf_class_weight,
                max_features=self.rf_max_features,
                random_state=42,
                n_jobs=-1
            )
            scale_flag = False

        elif self.model_family == "lightgbm":
            model = LGBMClassifier(
                n_estimators=self.lgbm_n_estimators,
                learning_rate=self.lgbm_learning_rate,
                num_leaves=self.lgbm_num_leaves,
                min_child_samples=self.lgbm_min_child_samples,
                subsample=self.lgbm_subsample,
                colsample_bytree=self.lgbm_colsample_bytree,
                class_weight=self.lgbm_class_weight,
                random_state=42
            )
            scale_flag = False

        elif self.model_family == "xgboost":
            model = XGBClassifier(
                n_estimators=self.xgb_n_estimators,
                learning_rate=self.xgb_learning_rate,
                max_depth=self.xgb_max_depth,
                min_child_weight=self.xgb_min_child_weight,
                subsample=self.xgb_subsample,
                colsample_bytree=self.xgb_colsample_bytree,
                scale_pos_weight=self.xgb_scale_pos_weight,
                eval_metric="logloss",
                random_state=42
            )
            scale_flag = False

        else:
            raise ValueError(f"Unknown model family: {self.model_family}")

        return model, scale_flag

    def fit(self, X, y):
        X = X.copy()

        model, scale_flag = self._build_model()

        use_vif_flag = (self.model_family == "logistic") and self.use_vif

        X_proc, X_model, artifacts = fit_preprocessing_pipeline(
            X_train=X,
            use_scaling=scale_flag,
            use_vif=use_vif_flag,
            vif_threshold=self.vif_threshold,
            winsorize=self.winsorize,
            lower_q=self.lower_q,
            upper_q=self.upper_q,
            variance_threshold=self.variance_threshold
        )

        model.fit(X_model, y)

        self.model_ = model
        self.scale_flag_ = scale_flag
        self.artifacts_ = artifacts
        self.feature_names_ = artifacts["final_columns"]

        return self

    def predict_proba(self, X):
        X = X.copy()
        _, X_model = apply_preprocessing_pipeline(
            X,
            artifacts=self.artifacts_,
            use_scaling=self.scale_flag_
        )
        return self.model_.predict_proba(X_model)

    def predict(self, X):
        prob = self.predict_proba(X)[:, 1]
        pred = (prob >= CLASSIFICATION_THRESHOLD).astype(int)
        return pred


# =========================================================
# STEP 13: BUSINESS RANKING HELPERS
# =========================================================
def build_decile_table(scored_df, prob_col, target_col, n_bins=10):
    tmp = scored_df[[prob_col, target_col]].copy()
    tmp = tmp.sort_values(prob_col, ascending=False).reset_index(drop=True)

    tmp["rank"] = np.arange(1, len(tmp) + 1)
    tmp["decile"] = pd.qcut(
        tmp["rank"],
        q=n_bins,
        labels=[f"D{i}" for i in range(1, n_bins + 1)]
    )

    overall_churn_rate = tmp[target_col].mean()

    decile_tbl = (
        tmp.groupby("decile", observed=False)
           .agg(
               customers=(target_col, "count"),
               actual_churners=(target_col, "sum"),
               avg_predicted_prob=(prob_col, "mean")
           )
           .reset_index()
    )

    decile_tbl["churn_rate"] = decile_tbl["actual_churners"] / decile_tbl["customers"]
    decile_tbl["overall_churn_rate"] = overall_churn_rate
    decile_tbl["lift"] = np.where(
        overall_churn_rate > 0,
        decile_tbl["churn_rate"] / overall_churn_rate,
        np.nan
    )

    decile_tbl["cum_customers"] = decile_tbl["customers"].cumsum()
    decile_tbl["cum_churners"] = decile_tbl["actual_churners"].cumsum()

    total_churners = decile_tbl["actual_churners"].sum()
    decile_tbl["cum_recall"] = np.where(
        total_churners > 0,
        decile_tbl["cum_churners"] / total_churners,
        np.nan
    )

    return decile_tbl


def recall_at_top_percent(scored_df, prob_col, target_col, top_pct=0.10):
    tmp = scored_df[[prob_col, target_col]].copy()
    tmp = tmp.sort_values(prob_col, ascending=False).reset_index(drop=True)

    n_total = len(tmp)
    n_top = max(1, int(np.ceil(n_total * top_pct)))

    top_slice = tmp.head(n_top)

    total_churners = tmp[target_col].sum()
    captured_churners = top_slice[target_col].sum()

    recall_top = captured_churners / total_churners if total_churners > 0 else np.nan
    precision_top = top_slice[target_col].mean() if len(top_slice) > 0 else np.nan
    baseline_rate = tmp[target_col].mean() if len(tmp) > 0 else np.nan
    lift_top = precision_top / baseline_rate if baseline_rate > 0 else np.nan

    return {
        "top_pct": top_pct,
        "n_total": n_total,
        "n_top": n_top,
        "total_churners": total_churners,
        "captured_churners": captured_churners,
        "recall_at_top_pct": recall_top,
        "precision_at_top_pct": precision_top,
        "baseline_churn_rate": baseline_rate,
        "lift_at_top_pct": lift_top
    }


# =========================================================
# STEP 14: CUSTOM SCORERS FOR GRIDSEARCHCV
# =========================================================
#
# PURPOSE
# ---------------------------------------------------------
# GridSearchCV can use custom scorer callables of the form:
# scorer(estimator, X, y)
#
# We define both ML metrics and business ranking metrics.
# Grid search will refit using lift_top10.
# =========================================================
def _safe_auc(y_true, prob):
    return roc_auc_score(y_true, prob) if pd.Series(y_true).nunique() > 1 else np.nan


def scorer_auc(estimator, X, y):
    prob = estimator.predict_proba(X)[:, 1]
    return _safe_auc(y, prob)


def scorer_recall(estimator, X, y):
    prob = estimator.predict_proba(X)[:, 1]
    pred = (prob >= CLASSIFICATION_THRESHOLD).astype(int)
    return recall_score(y, pred, zero_division=0)


def scorer_precision(estimator, X, y):
    prob = estimator.predict_proba(X)[:, 1]
    pred = (prob >= CLASSIFICATION_THRESHOLD).astype(int)
    return precision_score(y, pred, zero_division=0)


def scorer_f1(estimator, X, y):
    prob = estimator.predict_proba(X)[:, 1]
    pred = (prob >= CLASSIFICATION_THRESHOLD).astype(int)
    return f1_score(y, pred, zero_division=0)


def scorer_recall_top10(estimator, X, y):
    prob = estimator.predict_proba(X)[:, 1]
    tmp = pd.DataFrame({"prob": prob, "y": y})
    return recall_at_top_percent(tmp, "prob", "y", 0.10)["recall_at_top_pct"]


def scorer_precision_top10(estimator, X, y):
    prob = estimator.predict_proba(X)[:, 1]
    tmp = pd.DataFrame({"prob": prob, "y": y})
    return recall_at_top_percent(tmp, "prob", "y", 0.10)["precision_at_top_pct"]


def scorer_lift_top10(estimator, X, y):
    prob = estimator.predict_proba(X)[:, 1]
    tmp = pd.DataFrame({"prob": prob, "y": y})
    return recall_at_top_percent(tmp, "prob", "y", 0.10)["lift_at_top_pct"]


def scorer_recall_top20(estimator, X, y):
    prob = estimator.predict_proba(X)[:, 1]
    tmp = pd.DataFrame({"prob": prob, "y": y})
    return recall_at_top_percent(tmp, "prob", "y", 0.20)["recall_at_top_pct"]


def scorer_precision_top20(estimator, X, y):
    prob = estimator.predict_proba(X)[:, 1]
    tmp = pd.DataFrame({"prob": prob, "y": y})
    return recall_at_top_percent(tmp, "prob", "y", 0.20)["precision_at_top_pct"]


def scorer_lift_top20(estimator, X, y):
    prob = estimator.predict_proba(X)[:, 1]
    tmp = pd.DataFrame({"prob": prob, "y": y})
    return recall_at_top_percent(tmp, "prob", "y", 0.20)["lift_at_top_pct"]


scoring_dict = {
    "auc": scorer_auc,
    "recall": scorer_recall,
    "precision": scorer_precision,
    "f1": scorer_f1,
    "recall_top10": scorer_recall_top10,
    "precision_top10": scorer_precision_top10,
    "lift_top10": scorer_lift_top10,
    "recall_top20": scorer_recall_top20,
    "precision_top20": scorer_precision_top20,
    "lift_top20": scorer_lift_top20
}


# =========================================================
# STEP 15: PARAMETER GRID FOR EACH MODEL FAMILY
# =========================================================
#
# IMPORTANT
# ---------------------------------------------------------
# This is a true GridSearchCV parameter grid.
# It includes important hyperparameters and imbalance knobs.
#
# Keep this reasonably sized.
# Expanding CV + multiple models can get expensive quickly.
# =========================================================
param_grid = [
    # Logistic Regression
    {
        "model_family": ["logistic"],
        "C": [0.1, 1.0, 3.0],
        "logistic_class_weight": [None, "balanced"],
        "penalty": ["l2"],
        "use_vif": [False]
    },

    # Random Forest
    {
        "model_family": ["rf"],
        "rf_n_estimators": [200, 300],
        "rf_max_depth": [6, 8, None],
        "rf_min_samples_leaf": [5, 10],
        "rf_class_weight": [None, "balanced", "balanced_subsample"],
        "rf_max_features": ["sqrt"]
    },

    # LightGBM
    {
        "model_family": ["lightgbm"],
        "lgbm_n_estimators": [200, 300],
        "lgbm_learning_rate": [0.03, 0.05],
        "lgbm_num_leaves": [31, 63],
        "lgbm_min_child_samples": [20, 50],
        "lgbm_subsample": [0.8, 1.0],
        "lgbm_colsample_bytree": [0.8, 1.0],
        "lgbm_class_weight": [None, "balanced"]
    },

    # XGBoost
    {
        "model_family": ["xgboost"],
        "xgb_n_estimators": [200, 300],
        "xgb_learning_rate": [0.03, 0.05],
        "xgb_max_depth": [4, 6],
        "xgb_min_child_weight": [1, 5],
        "xgb_subsample": [0.8, 1.0],
        "xgb_colsample_bytree": [0.8, 1.0],
        "xgb_scale_pos_weight": [10, 20]
    }
]

print("\nTotal hyperparameter combinations to evaluate:",
      sum(len(list(ParameterGrid(g))) for g in param_grid))


# =========================================================
# STEP 16: BUILD PRETEST / OOS DATASETS FOR GRID SEARCH
# =========================================================
#
# PURPOSE
# ---------------------------------------------------------
# GridSearchCV must run ONLY on pretest months.
# OOS months must remain untouched for final evaluation.
# =========================================================
pretest_df = get_snapshot_slice(model_df, pretest_months).sort_values(
    ["snapshot_month", "account_number"]
).reset_index(drop=True)

oos_df = get_snapshot_slice(model_df, oos_months).sort_values(
    ["snapshot_month", "account_number"]
).reset_index(drop=True)

X_pretest = pretest_df[feature_cols].copy()
y_pretest = pretest_df[TARGET_COL].copy()

X_oos = oos_df[feature_cols].copy()
y_oos = oos_df[TARGET_COL].copy()

print("\nPretest shape:", pretest_df.shape)
print("OOS shape    :", oos_df.shape)


# =========================================================
# STEP 17: CONVERT EXPANDING MONTH FOLDS TO ROW-INDEX FOLDS
# FOR GRIDSEARCHCV
# =========================================================
#
# PURPOSE
# ---------------------------------------------------------
# GridSearchCV accepts cv as:
# - int
# - splitter object
# - iterable of (train_idx, val_idx)
#
# Here we pass a custom iterable of row-index splits that
# reflect your exact expanding walk-forward fold design.
# =========================================================
grid_cv_splits = []

for fold in cv_folds:
    train_idx = pretest_df.index[pretest_df["snapshot_month"].isin(fold["train_months"])].to_numpy()
    val_idx = pretest_df.index[pretest_df["snapshot_month"].isin(fold["val_months"])].to_numpy()

    grid_cv_splits.append((train_idx, val_idx))

print("\nNumber of GridSearchCV folds:", len(grid_cv_splits))
for i, (tr, va) in enumerate(grid_cv_splits, 1):
    print(f"Fold {i}: train_rows={len(tr)}, val_rows={len(va)}")


# =========================================================
# STEP 18: RUN GRIDSEARCHCV WITH CUSTOM TIME-BASED FOLDS
# =========================================================
#
# REFIT STRATEGY
# ---------------------------------------------------------
# We refit using lift_top10 because your use case is highly
# ranking / targeting oriented.
#
# You can change to auc if needed, but this keeps alignment
# with your business-first selection logic.
# =========================================================
base_estimator = ChurnPreprocessedEstimator(
    winsorize=True,
    lower_q=0.05,
    upper_q=0.95,
    variance_threshold=0.00,
    use_vif=False,          # keep disabled for now
    vif_threshold=10
)

grid_search = GridSearchCV(
    estimator=base_estimator,
    param_grid=param_grid,
    scoring=scoring_dict,
    refit="lift_top10",
    cv=grid_cv_splits,
    n_jobs=1,               # safer because custom estimator + external libs
    verbose=2,
    return_train_score=False
)

grid_search.fit(X_pretest, y_pretest)

print("\nBest params from GridSearchCV:")
print(json.dumps(grid_search.best_params_, indent=2, default=str))

print("\nBest lift_top10 score from GridSearchCV:")
print(grid_search.best_score_)


# =========================================================
# STEP 19: GRID SEARCH SUMMARY TABLE
# =========================================================
grid_results_df = pd.DataFrame(grid_search.cv_results_)

summary_cols = [
    "rank_test_lift_top10",
    "mean_test_auc",
    "mean_test_recall",
    "mean_test_precision",
    "mean_test_f1",
    "mean_test_recall_top10",
    "mean_test_precision_top10",
    "mean_test_lift_top10",
    "mean_test_recall_top20",
    "mean_test_precision_top20",
    "mean_test_lift_top20",
    "params"
]

summary_df = grid_results_df[summary_cols].copy()
summary_df = summary_df.sort_values(
    ["rank_test_lift_top10", "mean_test_recall_top10", "mean_test_auc"],
    ascending=[True, False, False]
).reset_index(drop=True)

print("\n===== GRID SEARCH SUMMARY =====")
print(summary_df.head(20))


# =========================================================
# STEP 20: FINAL BEST MODEL TRAINING ON PRETEST MONTHS
# =========================================================
#
# PURPOSE
# ---------------------------------------------------------
# GridSearchCV already refits best estimator on ALL pretest data
# because refit=True.
#
# So the final fitted model is:
# grid_search.best_estimator_
# =========================================================
final_model = grid_search.best_estimator_

print("\n===== FINAL BEST ESTIMATOR =====")
print(final_model)


# =========================================================
# STEP 21: OOS EVALUATION
# =========================================================
oos_prob = final_model.predict_proba(X_oos)[:, 1]
oos_pred = (oos_prob >= CLASSIFICATION_THRESHOLD).astype(int)

oos_auc = roc_auc_score(y_oos, oos_prob) if y_oos.nunique() > 1 else np.nan
oos_recall = recall_score(y_oos, oos_pred, zero_division=0)
oos_precision = precision_score(y_oos, oos_pred, zero_division=0)
oos_f1 = f1_score(y_oos, oos_pred, zero_division=0)

print("\n===== OOS PERFORMANCE =====")
print("OOS AUC:", oos_auc)
print("OOS Recall:", oos_recall)
print("OOS Precision:", oos_precision)
print("OOS F1:", oos_f1)


# =========================================================
# STEP 22: OOS SCORED OUTPUT
# =========================================================
oos_scored = oos_df[["account_number", "snapshot_month", TARGET_COL]].copy()
oos_scored["predicted_churn_probability_3m"] = oos_prob
oos_scored["predicted_churn_flag_3m"] = oos_pred

print("\nSample OOS scored output:")
print(oos_scored.head(20))


# =========================================================
# STEP 22A: OOS DECILE TABLE + TOP-K METRICS
# =========================================================
oos_eval_df = oos_scored.copy()

decile_table_oos = build_decile_table(
    scored_df=oos_eval_df,
    prob_col="predicted_churn_probability_3m",
    target_col=TARGET_COL,
    n_bins=10
)

print("\n===== OOS DECILE TABLE =====")
print(decile_table_oos)

top10_metrics = recall_at_top_percent(
    scored_df=oos_eval_df,
    prob_col="predicted_churn_probability_3m",
    target_col=TARGET_COL,
    top_pct=0.10
)

top20_metrics = recall_at_top_percent(
    scored_df=oos_eval_df,
    prob_col="predicted_churn_probability_3m",
    target_col=TARGET_COL,
    top_pct=0.20
)

print("\n===== OOS TOP-K METRICS =====")
print("Recall @ Top 10%:", top10_metrics["recall_at_top_pct"])
print("Precision @ Top 10%:", top10_metrics["precision_at_top_pct"])
print("Lift @ Top 10%:", top10_metrics["lift_at_top_pct"])

print("Recall @ Top 20%:", top20_metrics["recall_at_top_pct"])
print("Precision @ Top 20%:", top20_metrics["precision_at_top_pct"])
print("Lift @ Top 20%:", top20_metrics["lift_at_top_pct"])


# =========================================================
# STEP 23: LATEST SNAPSHOT PRODUCTION-STYLE SCORING
# =========================================================
latest_month = max(all_months_full)

latest_df = df[
    (df["snapshot_month"] == latest_month) &
    (df["lifecycle_stage_lower"] == "active") &
    (df["tenure_months"] >= MIN_TENURE_MONTHS)
].copy()

if not latest_df.empty:
    X_latest = latest_df[feature_cols].copy()

    latest_df["predicted_churn_probability_next_3m"] = final_model.predict_proba(X_latest)[:, 1]

    print("\nLatest snapshot production-style scoring:")
    print(
        latest_df[
            ["account_number", "snapshot_month", "predicted_churn_probability_next_3m"]
        ].head(20)
    )
else:
    print("\nNo latest active eligible customers found for production-style scoring.")


# =========================================================
# STEP 24: OPTIONAL FEATURE IMPORTANCE / COEFFICIENTS
# =========================================================
#
# IMPORTANT
# ---------------------------------------------------------
# The custom estimator stores:
# - final_model.model_        -> actual trained underlying model
# - final_model.feature_names_ -> post-preprocessing feature names
# =========================================================
underlying_model = final_model.model_
training_columns = final_model.feature_names_

if final_model.model_family in ["rf", "lightgbm", "xgboost"]:
    feature_importance_df = pd.DataFrame({
        "feature": training_columns,
        "importance": underlying_model.feature_importances_
    }).sort_values("importance", ascending=False)

    print("\nTop 25 feature importances:")
    print(feature_importance_df.head(25))

elif final_model.model_family == "logistic":
    coef_df = pd.DataFrame({
        "feature": training_columns,
        "coefficient": underlying_model.coef_[0]
    }).sort_values("coefficient", ascending=False)

    print("\nTop positive logistic coefficients:")
    print(coef_df.head(25))

    print("\nTop negative logistic coefficients:")
    print(coef_df.tail(25))

KeyboardInterrupt: 

In [ ]:
latest_df

,account_number,snapshot_month,household_composition,activation_date,lifecycle_stage,geographic_market,premises_type,value_tier,credit_profile,commitment_end_date,...,tenure_bucket_encoded,ticket_spike_ratio,outage_trend_ratio,usage_last_3m,usage_last_6m,usage_drop_ratio,fiber_speed_risk,service_frustration_index,frustration_acceleration,predicted_churn_probability_next_3m
15,200000,2025-12-01,Family,2023-02-01,Active,Metro,Bulk,Mid,A,NaT,...,1,0.500000,0.333333,275.933333,275.583333,1.001270,0.000000,1.320290,-0.168841,0.001028
39,200015,2025-12-01,Family,2023-12-01,Active,Rural,MDU,Mid,B,2025-11-12,...,1,0.500000,0.750000,143.400000,152.650000,0.939404,0.000000,1.845712,0.931126,0.000619
81,200023,2025-12-01,Single,2022-01-01,Active,Metro,MDU,Low,A,NaT,...,2,0.800000,0.000000,237.700000,205.833333,1.154818,0.347732,1.147732,0.004032,0.000435
95,200033,2025-12-01,Single,2024-09-01,Active,Rural,Bulk,Mid,B,NaT,...,0,0.400000,0.500000,240.466667,240.683333,0.999100,0.000000,1.271960,0.097854,0.000216
156,200044,2025-12-01,Family,2023-08-01,Active,Urban,MDU,Mid,D,2026-03-24,...,1,0.400000,0.571429,194.666667,214.050000,0.909445,0.502752,1.474181,-0.368174,0.000197
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4109,200786,2025-12-01,Roommates,2023-06-01,Active,Urban,SFH,Low,A,NaT,...,1,0.666667,0.500000,188.300000,203.133333,0.926977,0.388889,1.555556,0.527778,0.001594
4139,200791,2025-12-01,Single,2023-05-01,Active,Metro,MDU,Mid,A,2026-03-11,...,1,0.666667,0.750000,208.333333,204.133333,1.020575,0.493865,1.910532,0.765678,0.112031
4163,200792,2025-12-01,Roommates,2022-10-01,Active,Suburban,MDU,Low,C,2025-07-07,...,2,1.000000,0.000000,204.266667,231.683333,0.881663,0.000000,1.413158,0.762281,0.003746
4185,200797,2025-12-01,Family,2023-07-01,Active,Urban,MDU,Mid,B,2025-10-04,...,1,0.500000,0.833333,183.933333,172.666667,1.065251,0.502907,1.836240,-0.458333,0.000094


In [ ]:
# Lift Analysis : evaluates whether the highest-risk customers truly contain more churners than average.

oos_scored = oos_scored.sort_values(
    "predicted_churn_probability_3m",
    ascending=False
)

top20 = oos_scored.head(int(len(oos_scored)*0.2))

print("Top 20% churn rate:", top20["churn_flag_3m"].mean())
print("Overall churn rate:", oos_scored["churn_flag_3m"].mean())

Top 20% churn rate: 0.2077922077922078
Overall churn rate: 0.06426735218508997


In [ ]:
# Calculates standard classification metrics using the predicted churn probabilities

from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

y_true = oos_scored["churn_flag_3m"]
y_prob = oos_scored["predicted_churn_probability_3m"]
y_pred = oos_scored["predicted_churn_flag_3m"]

print("AUC:", roc_auc_score(y_true, y_prob))
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1:", f1_score(y_true, y_pred))

AUC: 0.8241758241758241
Precision: 0.45
Recall: 0.36
F1: 0.4


In [ ]:
# Groups customers into risk buckets based on predicted churn probability.

oos_scored["risk_decile"] = pd.qcut(
    oos_scored["predicted_churn_probability_3m"],
    10,
    labels=False
)

oos_scored.groupby("risk_decile").agg(
    avg_probability=("predicted_churn_probability_3m","mean"),
    churn_rate=("churn_flag_3m","mean"),
    customers=("account_number","nunique")
).sort_index(ascending=False)

,avg_probability,churn_rate,customers
risk_decile,,,
9,0.080099,0.333333,29
8,0.007185,0.076923,35
7,0.003298,0.076923,32
6,0.001957,0.025641,36
5,0.001114,0.052632,36
4,0.000725,0.000000,37
3,0.000483,0.051282,36
2,0.000310,0.025641,35
1,0.000186,0.000000,35


In [ ]:
grid_results_df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_logistic_class_weight,param_model_family,param_penalty,param_use_vif,param_rf_class_weight,...,split2_test_precision_top20,mean_test_precision_top20,std_test_precision_top20,rank_test_precision_top20,split0_test_lift_top20,split1_test_lift_top20,split2_test_lift_top20,mean_test_lift_top20,std_test_lift_top20,rank_test_lift_top20
0,0.061037,0.010522,0.244320,0.019976,0.1,NaN,logistic,l2,False,NaN,...,0.050,0.073984,0.033918,293,2.5,1.666667,1.633333,1.933333,0.400925,293
1,0.062018,0.003849,0.227357,0.020307,0.1,balanced,logistic,l2,False,NaN,...,0.025,0.057520,0.030096,296,2.0,1.666667,0.816667,1.494444,0.498207,296
2,0.056356,0.003180,0.214938,0.003671,1.0,NaN,logistic,l2,False,NaN,...,0.025,0.065650,0.041098,294,2.5,1.666667,0.816667,1.661111,0.687229,294
3,0.064909,0.002142,0.211901,0.004846,1.0,balanced,logistic,l2,False,NaN,...,0.025,0.057520,0.030096,296,2.0,1.666667,0.816667,1.494444,0.498207,296
4,0.063143,0.001754,0.213354,0.002805,3.0,NaN,logistic,l2,False,NaN,...,0.025,0.065650,0.041098,294,2.5,1.666667,0.816667,1.661111,0.687229,294
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
293,0.433074,0.051409,0.685299,0.140309,NaN,NaN,xgboost,NaN,NaN,NaN,...,0.050,0.115244,0.049769,44,3.5,4.166667,1.633333,3.100000,1.072208,53
294,0.565389,0.092120,0.720252,0.171390,NaN,NaN,xgboost,NaN,NaN,NaN,...,0.000,0.098577,0.072162,228,3.5,4.166667,0.000000,2.555556,1.827432,231
295,0.476211,0.024597,0.511095,0.030498,NaN,NaN,xgboost,NaN,NaN,NaN,...,0.050,0.123374,0.059257,26,4.0,4.166667,1.633333,3.266667,1.156944,26
296,0.496203,0.094206,0.716333,0.074131,NaN,NaN,xgboost,NaN,NaN,NaN,...,0.050,0.115244,0.049769,44,3.5,4.166667,1.633333,3.100000,1.072208,53


In [ ]:
final_model.model_

,boosting_type,'gbdt'
,num_leaves,63
,max_depth,-1
,learning_rate,0.05
,n_estimators,200
,subsample_for_bin,200000
,objective,None
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [ ]:
grid_search.best_estimator_

,model_family,'lightgbm'
,winsorize,True
,lower_q,0.05
,upper_q,0.95
,variance_threshold,0.0
,use_vif,False
,vif_threshold,10
,C,1.0
,logistic_class_weight,'balanced'
,penalty,'l2'
,rf_n_estimators,200


In [ ]:
# Actual Churn Rate - OOS
oos_scored["churn_flag_3m"].mean()

np.float64(0.06426735218508997)

In [ ]:
# Predicted Churn Rate - OOS
oos_scored["predicted_churn_flag_3m"].mean()

np.float64(0.05141388174807198)

In [ ]:
# Sort by predicted probability (descending)
tmp = oos_scored.sort_values(
    "predicted_churn_probability_3m",
    ascending=False
).reset_index(drop=True)


# ---------- TOP 10% ----------
top10_n = int(len(tmp) * 0.10)
top10 = tmp.head(top10_n)


recall_top10 = top10["churn_flag_3m"].sum() / tmp["churn_flag_3m"].sum()
precision_top10 = top10["churn_flag_3m"].mean()
lift_top10 = precision_top10 / tmp["churn_flag_3m"].mean()


print("=== TOP 10% ===")
print("Recall:", recall_top10)
print("Precision:", precision_top10)
print("Lift:", lift_top10)




# ---------- TOP 20% ----------
top20_n = int(len(tmp) * 0.20)
top20 = tmp.head(top20_n)


recall_top20 = top20["churn_flag_3m"].sum() / tmp["churn_flag_3m"].sum()
precision_top20 = top20["churn_flag_3m"].mean()
lift_top20 = precision_top20 / tmp["churn_flag_3m"].mean()


print("\n=== TOP 20% ===")
print("Recall:", recall_top20)
print("Precision:", precision_top20)
print("Lift:", lift_top20)


=== TOP 10% ===
Recall: 0.52
Precision: 0.34210526315789475
Lift: 5.323157894736842

=== TOP 20% ===
Recall: 0.64
Precision: 0.2077922077922078
Lift: 3.2332467532467537


In [40]:
df

,account_number,snapshot_month,household_composition,activation_date,lifecycle_stage,geographic_market,premises_type,value_tier,credit_profile,commitment_end_date,...,tenure_bucket,tenure_bucket_encoded,ticket_spike_ratio,outage_trend_ratio,usage_last_3m,usage_last_6m,usage_drop_ratio,fiber_speed_risk,service_frustration_index,frustration_acceleration
0,200000,2024-09-01,Family,2024-03-01,Active,Metro,Bulk,Mid,A,NaT,...,early_growth,0,1.000000,1.000000,292.700000,292.700000,1.000000,0.0,2.439130,0.000000
1,200000,2024-10-01,Family,2023-09-01,Active,Metro,Bulk,Mid,A,NaT,...,early_growth,0,1.000000,1.000000,257.650000,257.650000,1.000000,0.0,2.500000,0.000000
2,200000,2024-11-01,Family,2023-12-01,Active,Metro,Bulk,Mid,A,NaT,...,early_growth,0,1.000000,1.000000,254.966667,254.966667,1.000000,0.0,2.426087,0.000000
3,200000,2024-12-01,Family,2023-04-01,Active,Metro,Bulk,Mid,A,NaT,...,established,1,0.666667,0.800000,270.566667,276.100000,0.979959,0.0,1.918841,-0.520290
4,200000,2025-01-01,Family,2023-07-01,Active,Metro,Bulk,Mid,A,NaT,...,established,1,0.500000,0.714286,259.933333,259.020000,1.003526,0.0,1.655590,-0.844410
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4195,200799,2025-08-01,Family,2023-09-01,Active,Rural,MDU,High,C,NaT,...,established,1,0.250000,0.200000,238.866667,213.650000,1.118028,0.0,0.851515,-0.569697
4196,200799,2025-09-01,Family,2023-10-01,Active,Rural,MDU,High,C,NaT,...,established,1,0.500000,0.750000,252.033333,238.933333,1.054827,0.0,1.771212,0.831818
4197,200799,2025-10-01,Family,2024-03-01,Active,Rural,MDU,High,C,NaT,...,established,1,1.000000,1.000000,238.066667,244.133333,0.975150,0.0,2.445455,1.995455
4198,200799,2025-11-01,Family,2022-12-01,Active,Rural,MDU,High,C,NaT,...,established,1,0.750000,0.666667,213.000000,225.933333,0.942756,0.0,1.875758,1.024242


In [ ]:
# =========================================================
# STEP 23: LATEST CUSTOMER-LEVEL EXPLAINABLE CHURN OUTPUT
# ALIGNED TO GRIDSEARCHCV + WRAPPED FINAL MODEL
# =========================================================
#
# PURPOSE
# ---------------------------------------------------------
# This block creates the final production-style explainable output
# for the latest active eligible customers.
#
# It combines:
# 1. Customer-level churn probability
# 2. Top SHAP drivers explaining why the customer is risky
# 3. Automatic risk band assignment using Top-K ranking
#
# IMPORTANT INTERPRETATION
# ---------------------------------------------------------
# For customer-level "top drivers behind churn", we use ONLY
# POSITIVE SHAP values.
#
# So:
# - Positive SHAP value  -> pushes churn risk UP
# - Negative SHAP value  -> pushes churn risk DOWN
#
# This means the top 3 drivers shown below are true churn-
# increasing drivers, not just strongest influences overall.
#
# ALIGNMENT WITH GRIDSEARCHCV VERSION
# ---------------------------------------------------------
# In the new pipeline:
# - final_model is the wrapped estimator returned by GridSearchCV
# - final_model.model_ is the actual trained underlying model
# - final_model.artifacts_ stores preprocessing objects
# - final_model.feature_names_ stores post-preprocessing feature names
#
# Therefore:
# - scoring uses final_model.predict_proba(X_latest)
# - SHAP must be run on the PROCESSED version of X_latest
#   using the stored preprocessing artifacts
#
# OUTPUT
# ---------------------------------------------------------
# final_latest_risk_output
#
# Columns include:
# - account_number
# - snapshot_month
# - predicted_churn_probability_next_3m
# - risk_band
# - top_3_shap_drivers_str
# - top_3_shap_drivers_detailed_str
#
# RISK BAND DESIGN
# ---------------------------------------------------------
# - High Risk   = Top 10% highest-risk customers
# - Medium Risk = Next 20%
# - Low Risk    = Remaining customers
# =========================================================

try:
    import shap

    # -----------------------------------------------------
    # PART 0: IDENTIFY LATEST ACTIVE ELIGIBLE CUSTOMERS
    # -----------------------------------------------------
    latest_month = max(all_months_full)

    latest_df = df[
        (df["snapshot_month"] == latest_month) &
        (df["lifecycle_stage_lower"] == "active") &
        (df["tenure_months"] >= MIN_TENURE_MONTHS)
    ].copy()

    if not latest_df.empty:

        # -------------------------------------------------
        # PART 1: BUILD RAW LATEST FEATURE MATRIX
        # -------------------------------------------------
        X_latest = latest_df[feature_cols].copy()

        # -------------------------------------------------
        # PART 2: SCORE USING FINAL WRAPPED MODEL
        # -------------------------------------------------
        # In GridSearchCV version, final_model already handles
        # preprocessing internally during predict_proba().
        latest_df["predicted_churn_probability_next_3m"] = final_model.predict_proba(X_latest)[:, 1]

        # -------------------------------------------------
        # PART 3: PREPROCESS X_latest FOR SHAP
        # -------------------------------------------------
        # SHAP must explain the actual underlying model, so we
        # manually transform X_latest using stored train-fitted
        # preprocessing artifacts.
        #
        # final_model.artifacts_  -> preprocessing learned on pretest data
        # final_model.scale_flag_ -> whether scaling was used
        # -------------------------------------------------
        X_latest_proc, X_latest_model = apply_preprocessing_pipeline(
            X_latest,
            artifacts=final_model.artifacts_,
            use_scaling=final_model.scale_flag_
        )

        training_columns = final_model.feature_names_
        underlying_model = final_model.model_

        # -------------------------------------------------
        # PART 4: BUILD SHAP EXPLAINER BASED ON ACTUAL MODEL FAMILY
        # -------------------------------------------------
        if final_model.model_family in ["rf", "lightgbm", "xgboost"]:
            explainer = shap.TreeExplainer(underlying_model)
            shap_values_raw = explainer.shap_values(X_latest_proc)

            # Handle different SHAP output formats
            if isinstance(shap_values_raw, list):
                if len(shap_values_raw) == 2:
                    shap_values = shap_values_raw[1]
                else:
                    shap_values = shap_values_raw[0]
            elif isinstance(shap_values_raw, np.ndarray) and shap_values_raw.ndim == 3:
                shap_values = shap_values_raw[:, :, 1]
            else:
                shap_values = shap_values_raw

        elif final_model.model_family == "logistic":
            # For logistic, use the exact matrix the underlying model sees
            explainer = shap.LinearExplainer(underlying_model, X_latest_model)
            shap_values_raw = explainer.shap_values(X_latest_model)

            if isinstance(shap_values_raw, list):
                if len(shap_values_raw) == 2:
                    shap_values = shap_values_raw[1]
                else:
                    shap_values = shap_values_raw[0]
            else:
                shap_values = shap_values_raw

        else:
            raise ValueError(f"SHAP explainer not configured for model family: {final_model.model_family}")

        # -------------------------------------------------
        # PART 5: CONVERT SHAP VALUES TO DATAFRAME
        # -------------------------------------------------
        shap_df = pd.DataFrame(
            shap_values,
            columns=training_columns,
            index=latest_df.index
        )

        # -------------------------------------------------
        # PART 6: HELPER FUNCTIONS TO EXTRACT TOP POSITIVE DRIVERS
        # -------------------------------------------------
        def get_top_positive_shap_features(row, top_n=3):
            """
            Return top feature names that are increasing churn risk.
            Only positive SHAP values are considered.
            """
            positive_row = row[row > 0].sort_values(ascending=False)

            if len(positive_row) == 0:
                return ["No strong positive churn driver"]

            return positive_row.head(top_n).index.tolist()

        def get_top_positive_shap_features_detailed(row, top_n=3):
            """
            Return top churn-increasing feature names with SHAP values.
            Only positive SHAP values are considered.
            """
            positive_row = row[row > 0].sort_values(ascending=False)

            if len(positive_row) == 0:
                return ["No strong positive churn driver"]

            out = []
            for feat, val in positive_row.head(top_n).items():
                out.append(f"{feat} ({val:.4f})")
            return out

        # -------------------------------------------------
        # PART 7: ADD TOP POSITIVE SHAP DRIVERS TO latest_df
        # -------------------------------------------------
        latest_df["top_3_shap_drivers"] = shap_df.apply(
            lambda row: get_top_positive_shap_features(row, top_n=3),
            axis=1
        )

        latest_df["top_3_shap_drivers_detailed"] = shap_df.apply(
            lambda row: get_top_positive_shap_features_detailed(row, top_n=3),
            axis=1
        )

        latest_df["top_3_shap_drivers_str"] = latest_df["top_3_shap_drivers"].apply(
            lambda x: ", ".join(x)
        )

        latest_df["top_3_shap_drivers_detailed_str"] = latest_df["top_3_shap_drivers_detailed"].apply(
            lambda x: ", ".join(x)
        )

        # -------------------------------------------------
        # PART 8: CREATE AUTOMATED RISK BANDS USING TOP-K
        # -------------------------------------------------
        # latest_df = latest_df.sort_values(
        #     "predicted_churn_probability_next_3m",
        #     ascending=False
        # ).reset_index(drop=True)

        # latest_df["score_rank_pct"] = (latest_df.index + 1) / len(latest_df)

        # def assign_risk_band(rank_pct):
        #     if rank_pct <= 0.10:
        #         return "High Risk"
        #     elif rank_pct <= 0.30:
        #         return "Medium Risk"
        #     else:
        #         return "Low Risk"

        # latest_df["risk_band"] = latest_df["score_rank_pct"].apply(assign_risk_band)

        latest_df = latest_df.sort_values(
            "predicted_churn_probability_next_3m",
            ascending=False
        ).reset_index(drop=True)

        latest_df["score_rank_pct"] = (latest_df.index + 1) / len(latest_df)

        def assign_risk_band(rank_pct):
            if rank_pct <= 0.05:
                return "Very High Risk"
            elif rank_pct <= 0.15:
                return "High Risk"
            elif rank_pct <= 0.30:
                return "Medium Risk"
            else:
                return "Low Risk"

        latest_df["risk_band"] = latest_df["score_rank_pct"].apply(assign_risk_band)

        # -------------------------------------------------
        # PART 9: FINAL BUSINESS-FRIENDLY OUTPUT
        # -------------------------------------------------
        final_latest_risk_output = latest_df[
            [
                "account_number",
                "snapshot_month",
                "predicted_churn_probability_next_3m",
                "risk_band",
                "top_3_shap_drivers_str",
                "top_3_shap_drivers_detailed_str",
                "geographic_market",
                "household_composition",
                "premises_type",
                "tenure_bucket"
            ]
        ].copy()

        # -------------------------------------------------
        # PART 10: PRINT OUTPUTS
        # -------------------------------------------------
        print("\n===== FINAL LATEST CUSTOMER RISK OUTPUT =====")
        print(final_latest_risk_output.head(20))

        print("\n===== RISK BAND DISTRIBUTION =====")
        print(final_latest_risk_output["risk_band"].value_counts(dropna=False))

        print("\n===== AVG CHURN PROBABILITY BY RISK BAND =====")
        print(
            final_latest_risk_output.groupby("risk_band")[
                "predicted_churn_probability_next_3m"
            ].mean().sort_values(ascending=False)
        )

    else:
        print("\nSTEP 23 skipped because latest_df is empty.")

except ImportError:
    print("\nSHAP is not installed. Please run: pip install shap")
except Exception as e:
    print(f"\nSTEP 23 failed: {e}")

c:\Users\shreyaan.saha\Desktop\Projects\Actual Projs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



===== FINAL LATEST CUSTOMER RISK OUTPUT =====
    account_number snapshot_month  predicted_churn_probability_next_3m  \
0           200513     2025-12-01                             0.395172   
1           200791     2025-12-01                             0.112031   
2           200516     2025-12-01                             0.102801   
3           200244     2025-12-01                             0.099932   
4           200526     2025-12-01                             0.099856   
5           200705     2025-12-01                             0.089298   
6           200411     2025-12-01                             0.066544   
7           200389     2025-12-01                             0.058557   
8           200131     2025-12-01                             0.057724   
9           200595     2025-12-01                             0.049284   
10          200088     2025-12-01                             0.042637   
11          200061     2025-12-01                             0.0

c:\Users\shreyaan.saha\Desktop\Projects\Actual Projs\.venv\Lib\site-packages\shap\explainers\_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [ ]:
final_latest_risk_output

,account_number,snapshot_month,predicted_churn_probability_next_3m,risk_band,top_3_shap_drivers_str,top_3_shap_drivers_detailed_str,geographic_market,household_composition,premises_type,tenure_bucket
0,200513,2025-12-01,0.395172,Very High Risk,"usage_drop_ratio_3m_vs_12m, csat_6m_min, usage...","usage_drop_ratio_3m_vs_12m (1.0782), csat_6m_m...",Suburban,Family,SFH,established
1,200791,2025-12-01,0.112031,Very High Risk,"revenue_per_speed, csat_12m_min, bill_volatili...","revenue_per_speed (2.9955), csat_12m_min (0.77...",Metro,Single,MDU,established
2,200516,2025-12-01,0.102801,Very High Risk,"outage_severity_last_12m_sum, revenue_per_spee...","outage_severity_last_12m_sum (1.3299), revenue...",Urban,Single,SFH,long_term
3,200244,2025-12-01,0.099932,Very High Risk,"revenue_per_speed, bill_volatility_12m_std, cs...","revenue_per_speed (3.4322), bill_volatility_12...",Suburban,Single,MDU,established
4,200526,2025-12-01,0.099856,Very High Risk,"revenue_per_speed, bill_volatility_12m_std, us...","revenue_per_speed (2.4427), bill_volatility_12...",Suburban,Single,MDU,established
...,...,...,...,...,...,...,...,...,...,...
175,200797,2025-12-01,0.000094,Low Risk,"nps_3m_min, nps_6m_min, csat_12m_min","nps_3m_min (0.1871), nps_6m_min (0.1589), csat...",Urban,Family,MDU,established
176,200363,2025-12-01,0.000086,Low Risk,"csat_12m_min, outage_severity_last_12m_sum, np...","csat_12m_min (0.3601), outage_severity_last_12...",Rural,Family,MDU,established
177,200519,2025-12-01,0.000074,Low Risk,"ticket_per_active_month_12m, outages_last_6m_s...","ticket_per_active_month_12m (0.5022), outages_...",Suburban,Family,SFH,established
178,200733,2025-12-01,0.000048,Low Risk,"outage_severity_last_12m_sum, outage_severity,...","outage_severity_last_12m_sum (0.1209), outage_...",Rural,Single,SFH,established


In [ ]:
final_latest_risk_output.to_excel('FinalChurnOutput_SHAP_GS.xlsx')

In [ ]:
!pip install groq
import os
os.environ["GROQ_API_KEY"] = "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"

In [ ]:
# import plotly.io as pio
# pio.renderers.default = "browser"

In [ ]:
# =========================================================
# STEP 24: FINAL SINGLE-COLUMN CUSTOMER RECOMMENDATION ENGINE
# CLEAN VERSION WITH LOW-RISK RULES + CLEAN MERGE
# =========================================================
#
# PURPOSE
# ---------------------------------------------------------
# Create one business-friendly recommendation column per customer
# for easy Excel export and retention team consumption.
#
# DESIGN
# ---------------------------------------------------------
# - High / Medium Risk customers:
#     Use LLM if available
# - Low Risk customers:
#     Use rule-based recommendations
#
# WHY RULES FOR LOW RISK?
# ---------------------------------------------------------
# Low-risk customers usually do not require expensive or complex
# intervention. A deterministic low-cost monitoring / light-touch
# recommendation is generally sufficient and more efficient than
# calling an LLM for every low-risk customer.
#
# OUTPUT
# ---------------------------------------------------------
# final_latest_recommendation_output
#
# MAIN NEW COLUMN
# ---------------------------------------------------------
# - final_recommendation
#
# FINAL EXCEL FILE
# ---------------------------------------------------------
# final_latest_customer_recommendations.xlsx
# =========================================================


try:
    import os
    import json
    import time
    import pandas as pd


    USE_LLM = True
    MODEL_NAME = "moonshotai/kimi-k2-instruct-0905"
    TEMPERATURE = 0.2
    SLEEP_SEC = 0.3
    MAX_ROWS_TO_PROCESS = None   # set to 10 or 20 for testing


    if "final_latest_risk_output" not in globals():
        raise ValueError("final_latest_risk_output not found. Please run Step 23 first.")


    work_df = final_latest_risk_output.copy()


    if MAX_ROWS_TO_PROCESS is not None:
        work_df = work_df.head(MAX_ROWS_TO_PROCESS).copy()


    # -----------------------------------------------------
    # PART 1: OPTIONAL LLM CLIENT
    # -----------------------------------------------------
    client = None
    groq_api_key = os.environ.get("GROQ_API_KEY")


    if USE_LLM and groq_api_key:
        from groq import Groq
        client = Groq(api_key=groq_api_key)
        llm_available = True
    else:
        llm_available = False
        print("LLM not available. Using rule-based recommendations for all customers.")


    # -----------------------------------------------------
    # PART 2: DETAILED SYSTEM PROMPT
    # -----------------------------------------------------
    SYSTEM_PROMPT = """
You are a senior telecom retention strategist for a broadband / fixed-line service provider.


You are given customer-level churn risk information from a machine learning churn model for Copper DSL customers.
Your task is to generate a single, practical, business-friendly retention recommendation for each customer.


BUSINESS CONTEXT:
- The customer is currently active.
- The model predicts probability of churn in the next 3 months.
- The customer has been assigned a risk band such as High Risk, Medium Risk, or Low Risk.
- The top churn drivers are derived from SHAP and represent the strongest factors increasing churn risk.


YOUR GOAL:
Generate ONE clear recommendation that a retention or service team can understand immediately.


IMPORTANT INTERPRETATION RULES:
1. If drivers indicate service issues, outages, repeated complaints, frustration, low CSAT, or low NPS:
   - prioritize service recovery first
   - recommend proactive service outreach, diagnostics, repair, issue resolution, or service assurance
   - do NOT prioritize discount as the first action unless pricing is also clearly a major issue


2. If drivers indicate pricing, billing, bill shock, price increase, revenue-per-speed mismatch, or poor value perception:
   - recommend plan optimization, pricing review, right-sizing, value communication, renewal discussion, or targeted retention offer


3. If drivers indicate usage drop, disengagement, low activity, or shrinking consumption:
   - recommend re-engagement, digital nudges, plan-fit review, usage education, or proactive check-in


4. If drivers indicate contract-end / commitment-end timing:
   - recommend renewal outreach, loyalty upgrade, or early renewal save offer


5. If multiple driver types are present:
   - prioritize service issues first if they exist
   - then mention pricing/value if relevant
   - then mention engagement or renewal as secondary considerations


6. High Risk customers:
   - require immediate or high-priority intervention
   - recommendation should be stronger and more proactive


7. Medium Risk customers:
   - require proactive but more measured intervention


8. Low Risk customers:
   - do not recommend expensive manual intervention unless drivers strongly justify it
   - monitoring or light-touch digital retention is acceptable


OUTPUT STYLE:
- Return a single concise recommendation in plain business English
- It should read like a final action note for a CRM or Excel file
- It should mention:
  a) what should be done
  b) through which broad channel if relevant
  c) why, based on churn drivers
- Do not mention SHAP explicitly
- Do not mention machine learning
- Do not output JSON
- Do not output bullet points
- Do not output multiple options
- Output exactly one recommendation sentence or short paragraph
""".strip()


    # -----------------------------------------------------
    # PART 3: DRIVER TYPE CLASSIFICATION
    # -----------------------------------------------------
    def derive_driver_types(top_driver_text):
        if pd.isna(top_driver_text) or str(top_driver_text).strip() == "":
            return ["mixed_other"], "mixed_other"


        drivers = [x.strip().lower() for x in str(top_driver_text).split(",") if x.strip() != ""]


        service_keywords = [
            "ticket", "outage", "csat", "nps", "frustration",
            "service_frustration_index", "outage_severity", "repeat_issue"
        ]
        pricing_keywords = [
            "revenue", "bill", "price", "bill_shock", "price_increase",
            "late_payment", "collections"
        ]
        engagement_keywords = [
            "usage", "drop", "usage_mom", "engagement"
        ]
        contract_keywords = [
            "contract", "commitment", "near_contract_end", "months_to_commitment_end"
        ]


        category_counts = {
            "service_issue": 0,
            "pricing_value": 0,
            "engagement_drop": 0,
            "contract_renewal": 0
        }


        for drv in drivers:
            if any(k in drv for k in service_keywords):
                category_counts["service_issue"] += 1
            if any(k in drv for k in pricing_keywords):
                category_counts["pricing_value"] += 1
            if any(k in drv for k in engagement_keywords):
                category_counts["engagement_drop"] += 1
            if any(k in drv for k in contract_keywords):
                category_counts["contract_renewal"] += 1


        driver_types = [k for k, v in category_counts.items() if v > 0]


        if not driver_types:
            return ["mixed_other"], "mixed_other"


        primary_driver_type = max(category_counts, key=category_counts.get)
        return driver_types, primary_driver_type


    work_df[["driver_types", "primary_driver_type"]] = work_df["top_3_shap_drivers_str"].apply(
        lambda x: pd.Series(derive_driver_types(x))
    )


    # -----------------------------------------------------
    # PART 4: RULE-BASED SINGLE RECOMMENDATION
    # -----------------------------------------------------
    def build_rule_based_recommendation(row):
        risk = str(row.get("risk_band", "Low Risk"))
        driver_types = row.get("driver_types", ["mixed_other"])


        if isinstance(driver_types, str):
            driver_types = [driver_types]


        has_service = "service_issue" in driver_types
        has_pricing = "pricing_value" in driver_types
        has_engagement = "engagement_drop" in driver_types
        has_contract = "contract_renewal" in driver_types


        # Low-risk logic: light-touch, low-cost action
        if risk == "Low Risk":
            return "Monitor with digital nudges only."


        # if has_service and has_pricing:
        #     return (
        #         "Immediate proactive service recovery outreach is recommended to address experience-related issues first, "
        #         "and once service is stabilized, review pricing or plan fit to reduce near-term churn risk."
        #     )


        # if has_service:
        #     return (
        #         "Immediate service recovery outreach through call center or support follow-up is recommended to address "
        #         "service-quality concerns and improve customer experience before considering any commercial retention step."
        #     )


        # if has_pricing and has_contract:
        #     return (
        #         "Proactive retention outreach is recommended with a renewal-focused pricing or plan optimization discussion, "
        #         "as value perception and contract timing appear to be the main churn drivers."
        #     )


        # if has_pricing:
        #     return (
        #         "A targeted retention conversation focused on pricing review, plan right-sizing, or value communication is recommended, "
        #         "as pricing sensitivity appears to be driving churn risk."
        #     )


        # if has_engagement:
        #     return (
        #         "Place the customer into a proactive re-engagement journey with usage nudges and plan-fit review, "
        #         "as declining usage signals reduced engagement and possible product mismatch."
        #     )


        # if has_contract:
        #     return (
        #         "A proactive renewal outreach is recommended to address possible contract-end churn risk and reinforce loyalty before renewal."
        #     )


        # if risk == "High Risk":
        #     return (
        #         "Immediate manual review and outbound retention outreach are recommended, as the customer is high risk and shows multiple churn signals "
        #         "without a single dominant intervention theme."
        #     )


        # return (
        #     "A proactive retention review is recommended through a moderate-touch outreach, "
        #     "as the customer shows emerging churn signals that warrant monitoring and targeted intervention."
        # )


    # -----------------------------------------------------
    # PART 5: LLM PAYLOAD BUILDER
    # -----------------------------------------------------
    def build_llm_payload(row):
        payload = {
            "account_number": row.get("account_number"),
            "snapshot_month": str(row.get("snapshot_month")),
            "predicted_churn_probability_next_3m": row.get("predicted_churn_probability_next_3m"),
            "risk_band": row.get("risk_band"),
            "driver_types": row.get("driver_types"),
            "primary_driver_type": row.get("primary_driver_type"),
            "top_3_shap_drivers_str": row.get("top_3_shap_drivers_str"),
            "top_3_shap_drivers_detailed_str": row.get("top_3_shap_drivers_detailed_str"),
        }


        optional_cols = [
            "geographic_market",
            "household_composition",
            "premises_type",
            "tenure_bucket"
        ]


        for col in optional_cols:
            if col in row.index and pd.notna(row[col]):
                payload[col] = row[col]


        return payload


    # -----------------------------------------------------
    # PART 6: LLM CALL
    # -----------------------------------------------------
    def build_llm_recommendation(row):
        payload = build_llm_payload(row)

        user_prompt = f"""
    You are assisting a telecom retention team.

    Customer context:
    {json.dumps(payload, ensure_ascii=False, indent=2)}

    Task:
    Write exactly ONE retention recommendation of MAXIMUM 5 WORDS.

    Requirements:
    - ≤5 words total.
    - Plain business English.
    - Implies: [Channel/Action/Reason]
    - Prioritize service recovery over pricing.
    - No SHAP/AI/models mention.

    Return ONLY the recommendation text.
    """.strip()

        response = client.chat.completions.create(
            model=MODEL_NAME,
            temperature=TEMPERATURE,
            max_tokens=15,  # Hard cap for 5 words
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt}
            ]
        )

        rec_text = response.choices[0].message.content.strip()
        rec_text = rec_text.replace("```", "").strip().strip('"').strip()
        return rec_text


    # -----------------------------------------------------
    # PART 7: GENERATE RECOMMENDATION FOR EVERY CUSTOMER
    # -----------------------------------------------------
    final_recommendations = []


    print("\n===== GENERATING FINAL RECOMMENDATIONS =====")
    print("Rows to process:", len(work_df))
    print("LLM available:", llm_available)


    for i, (_, row) in enumerate(work_df.iterrows(), start=1):
        risk = row["risk_band"]


        try:
            if llm_available and risk in ["Very High Risk", "High Risk", "Medium Risk"]:
                recommendation = build_llm_recommendation(row)
                source = "llm"
                time.sleep(SLEEP_SEC)
            else:
                recommendation = build_rule_based_recommendation(row)
                source = "rule_based"


        except Exception as e:
            recommendation = build_rule_based_recommendation(row)
            source = f"rule_based_fallback_due_to_llm_error: {str(e)}"


        # IMPORTANT:
        # Keep only account_number + final_recommendation here
        # to avoid duplicate _x / _y columns after merge
        final_recommendations.append({
            "account_number": row["account_number"],
            "final_recommendation": recommendation,
            "recommendation_source":source
        })


        if i % 10 == 0 or i == len(work_df):
            print(f"Processed {i}/{len(work_df)} customers")


    # -----------------------------------------------------
    # PART 8: FINAL SIMPLE OUTPUT FOR EXCEL (CLEAN VERSION)
    # -----------------------------------------------------
    # Convert driver_types list to readable string inside work_df itself
    work_df["driver_types"] = work_df["driver_types"].apply(
        lambda x: ", ".join(x) if isinstance(x, list) else str(x)
    )


    # Recommendation dataframe only keeps account_number + recommendation
    recommendation_df = pd.DataFrame(final_recommendations)


    # Merge only one new column back
    final_latest_recommendation_output = work_df.merge(
        recommendation_df,
        on="account_number",
        how="left"
    )


    # Keep only clean business-friendly columns
    final_latest_recommendation_output = final_latest_recommendation_output[
        [
            "account_number",
            "snapshot_month",
            "predicted_churn_probability_next_3m",
            "risk_band",
            "top_3_shap_drivers_str",
            "primary_driver_type",
            "driver_types",
            "final_recommendation",
            "recommendation_source"
        ]
    ].copy()


    print("\n===== FINAL RECOMMENDATION OUTPUT =====")
    print(final_latest_recommendation_output.head(20))


    # -----------------------------------------------------
    # PART 9: EXPORT TO EXCEL
    # -----------------------------------------------------
    output_excel_name = "final_latest_customer_recommendations.xlsx"
    final_latest_recommendation_output.to_excel(output_excel_name, index=False)


    print(f"\nExcel file exported successfully: {output_excel_name}")


except Exception as e:
    print(f"\nSTEP 23A failed: {e}")


===== GENERATING FINAL RECOMMENDATIONS =====
Rows to process: 180
LLM available: True
Processed 10/180 customers
Processed 20/180 customers
Processed 30/180 customers
Processed 40/180 customers
Processed 50/180 customers
Processed 60/180 customers
Processed 70/180 customers
Processed 80/180 customers
Processed 90/180 customers
Processed 100/180 customers
Processed 110/180 customers
Processed 120/180 customers
Processed 130/180 customers
Processed 140/180 customers
Processed 150/180 customers
Processed 160/180 customers
Processed 170/180 customers
Processed 180/180 customers

===== FINAL RECOMMENDATION OUTPUT =====
    account_number snapshot_month  predicted_churn_probability_next_3m  \
0           200513     2025-12-01                             0.395172   
1           200791     2025-12-01                             0.112031   
2           200516     2025-12-01                             0.102801   
3           200244     2025-12-01                             0.099932   
4      